In [ ]:
import os
import sys

sys.path.insert(0, os.path.dirname(os.getcwd()))

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchmetrics.text import WordErrorRate
from torchmetrics.classification import Accuracy
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

from src.model import register_whisper_accent
from src.train.dataset import DataCollatorSpeechSeq2SeqWithPadding, WhisperDataset

register_whisper_accent()

In [ ]:
model_path = "openai/whisper-tiny.en"
is_multilingual = False
processor = AutoProcessor.from_pretrained(model_path)

In [ ]:
test_dataset = WhisperDataset(
    data_path="westbrook/English_Accent_DataSet",
    split="test",
    processor=processor,
    multilingual_model=is_multilingual,
    shuffle=False,
)


# Create data collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# Create DataLoader
dataloader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0,  # Set to >0 for multiprocessing
)

# Test the dataloader
batch = next(iter(dataloader))
print("Batch keys:", batch.keys())
print("Input features shape:", batch["input_features"].shape)
print("Labels shape:", batch["labels"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)

In [ ]:
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_path, device_map="auto", dtype=torch.bfloat16
)
if model.generation_config.is_multilingual:
    # model.generation_config.language = "hi"
    model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

In [ ]:
batch = {k: v.to(model.device) for k, v in batch.items()}
batch["input_features"] = torch.rand_like(batch["input_features"]).to(torch.bfloat16)

In [ ]:
out = model.generate(**batch)

In [ ]:
init_tokens = model._retrieve_init_tokens(
    batch["input_features"],
    batch["input_features"].shape[0],
    model.generation_config,
    model.config,
    3000,
    {},
)

In [ ]:
processor.decode(init_tokens)

In [ ]:
text_pred = processor.decode(
    out,
    # skip_special_tokens=True
)
# text_pred = [processor.tokenizer.normalize(t) for t in text_pred]
text_pred

In [ ]:
text_target = processor.decode(
    batch["labels"].masked_fill(
        batch["labels"] == -100, processor.tokenizer.eos_token_id
    ),
    skip_special_tokens=True,
)
text_target

In [ ]:
metric = WordErrorRate()
metric(text_pred, text_target)